# Batched ELO Tournament — 19×13, 21×15

Uses `run_tournament_megabatch` from `tournament_batched.py`.  
All matchup games run in parallel mega-batches.  
10 games per perspective, 32 sims, temperature=0.

In [ ]:
import os, re, pickle, types, sys, json, glob, time
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ or 'google.colab' in str(globals())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
if IN_COLAB:
    try:
        import jax
        if 'TPU' in str(jax.devices()):
            !pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
        else:
            raise Exception('No TPU')
    except:
        !pip install -q jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    !pip install -q flax optax mctx

import jax
import jax.numpy as jnp
print(f"JAX {jax.__version__}, devices: {jax.devices()}")

REPO_URL = "https://github.com/echoname6/phutball-jax.git"
REPO_DIR = "/content/phutball-jax" if IN_COLAB else "./phutball-jax"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print('Repo ready')

In [ ]:
from phutball_env_jax import EnvConfig
from network import create_transformer_network
from tournament_batched import run_tournament_megabatch
print('Imports OK — using run_tournament_megabatch')

In [ ]:
# === CONFIG ===
ROOT = "/content/drive/MyDrive/phutball_checkpoints"
D, L = 512, 12

GAMES_PER_PERSP = 10
NUM_SIMS = 32
MAX_BATCH = 128  # smaller for larger boards (memory)

MY_BOARDS = ['19x13', '21x15']

BABEL_DIRS = {
    '19x13': os.path.join(ROOT, f"transformer_19x13_d{D}_l{L}"),
    '21x15': os.path.join(ROOT, f"transformer_21x15_d{D}_l{L}"),
}

BABEL_RANGES = {
    '19x13': (231, 279),
    '21x15': (281, 529),
}

BLANK_DIRS = {
    '19x13': os.path.join(ROOT, f"blank_19x13_d{D}_l{L}"),
    '21x15': os.path.join(ROOT, f"blank_21x15_d{D}_l{L}"),
}

# Verify paths
for label, dirs in [('Babel', BABEL_DIRS), ('Blank', BLANK_DIRS)]:
    for board, path in dirs.items():
        exists = 'EXISTS' if os.path.isdir(path) else 'NOT FOUND'
        print(f"{label} {board}: {exists}")

# ForgivingUnpickler
class _StubModule(types.ModuleType):
    def __getattr__(self, name):
        return type(name, (), {
            '__init__': lambda self, *a, **kw: self.__dict__.update(kw),
            '__setstate__': lambda self, state: self.__dict__.update(
                state if isinstance(state, dict) else {}),
        })

class _FU(pickle.Unpickler):
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except (ModuleNotFoundError, AttributeError):
            if module not in sys.modules:
                sys.modules[module] = _StubModule(module)
            return getattr(sys.modules[module], name)

def scan_dir(directory):
    pat = re.compile(r'^checkpoint_(\d{6})\.pkl$')
    results = []
    if not os.path.isdir(directory): return results
    for f in os.listdir(directory):
        m = pat.match(f)
        if m: results.append((int(m.group(1)), os.path.join(directory, f)))
    return sorted(results)

def load_params(path):
    with open(path, 'rb') as f:
        ckpt = _FU(f).load()
    if not isinstance(ckpt, dict): ckpt = ckpt.__dict__
    params = ckpt.get('params', ckpt.get('network_params'))
    del ckpt  # free the buffer, opt_state, etc
    return {'network_params': params}

print('\nConfig OK')

In [ ]:
# === COLLECT AGENTS ===
all_agents = {}

for board in MY_BOARDS:
    agents = []
    
    if board in BABEL_DIRS and board in BABEL_RANGES:
        lo, hi = BABEL_RANGES[board]
        for it, path in scan_dir(BABEL_DIRS[board]):
            if lo <= it <= hi:
                agents.append((f'Babel_{it}', path))
    
    if board in BLANK_DIRS:
        for it, path in scan_dir(BLANK_DIRS[board]):
            agents.append((f'Blank_{it}', path))
    
    all_agents[board] = agents
    print(f"\n{board}: {len(agents)} agents")
    for name, _ in agents:
        print(f"  {name}")

In [ ]:
# === RUN TOURNAMENTS ===
tournament_results = {}

for board in MY_BOARDS:
    agents_raw = all_agents[board]
    if len(agents_raw) < 2:
        print(f'\n{board}: <2 agents, skipping'); continue
    
    rows, cols = int(board.split('x')[0]), int(board.split('x')[1])
    env_config = EnvConfig(rows=rows, cols=cols)
    network = create_transformer_network(
        rows=rows, cols=cols, d_model=D, n_layers=L,
        n_heads=4, ffn_dim=D*2, pos_encoding='goal_distance')
    
    print(f'\nLoading {board} params...')
    agents = []
    for name, path in agents_raw:
        print(f'  {name}...', end=' ')
        params = load_params(path)
        agents.append((name, params))
        print('OK')
    
    rng = jax.random.PRNGKey(42)
    t0 = time.time()
    results, elo = run_tournament_megabatch(
        agents=agents, network=network, env_config=env_config, rng=rng,
        games_per_perspective=GAMES_PER_PERSP, max_batch_size=MAX_BATCH,
        num_simulations=NUM_SIMS)
    print(f'\n{board} tournament: {time.time()-t0:.0f}s total')
    
    tournament_results[board] = {
        'results': results, 'elo': elo,
        'ranked': sorted(elo.items(), key=lambda x: -x[1]),
        'agents': [n for n, _ in agents],
    }

In [ ]:
# === PLOT: ELO RANKINGS ===
n_t = len(tournament_results)
if n_t > 0:
    fig, axes = plt.subplots(1, n_t, figsize=(5*n_t, 5))
    if n_t == 1: axes = [axes]
    for ax, (board, data) in zip(axes, tournament_results.items()):
        ranked = data['ranked']
        names = [r[0] for r in ranked]
        elos = [r[1] for r in ranked]
        colors = ['#d9b357' if 'Babel' in n else '#4a7fb5' for n in names]
        bars = ax.barh(range(len(names)), elos, color=colors, alpha=0.85, edgecolor='#333')
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
        ax.set_xlabel('ELO'); ax.set_title(f'{board} Tournament')
        ax.axvline(1500, color='gray', ls=':', alpha=0.4); ax.invert_yaxis()
        ax.grid(True, alpha=0.2, axis='x')
        for bar, e in zip(bars, elos):
            ax.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
                    f'{e:.0f}', ha='left', va='center', fontsize=6)
    plt.tight_layout()
    fig.savefig('elo_rankings_19_21.pdf', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# === PLOT: ELO PROGRESSION ===
if n_t > 0:
    fig, axes = plt.subplots(1, n_t, figsize=(5*n_t, 4))
    if n_t == 1: axes = [axes]
    for ax, (board, data) in zip(axes, tournament_results.items()):
        elo = data['elo']
        for pfx, c, mk in [('Babel', '#d9b357', 'o'), ('Blank', '#4a7fb5', 's')]:
            pts = sorted([(int(re.search(r'\d+', n).group()), elo[n])
                           for n in elo if n.startswith(pfx)])
            if pts:
                ax.plot(*zip(*pts), label=pfx, color=c, marker=mk, markersize=4, lw=1.5)
        ax.axhline(1500, color='gray', ls=':', alpha=0.4)
        ax.set_xlabel('Iteration'); ax.set_ylabel('ELO')
        ax.set_title(f'{board} — ELO Progression')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig('elo_progression_19_21.pdf', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# === GAME STATS + BABEL vs BLANK ===
print('Game statistics from tournament matches:')
print('='*60)
for board, data in tournament_results.items():
    total_games = sum(r['total_games'] for r in data['results'])
    avg_lens = [r['avg_game_length'] for r in data['results'] if r['avg_game_length'] > 0]
    avg_jumps = [r['avg_jumps_per_game'] for r in data['results'] if r['avg_jumps_per_game'] > 0]
    print(f"\n  {board}: {total_games} games")
    if avg_lens: print(f"    avg game length: {np.mean(avg_lens):.1f}")
    if avg_jumps: print(f"    avg jumps/game: {np.mean(avg_jumps):.1f}")

print(f"\n{'='*60}")
print('Babel vs Blank — best agent per board:')
print(f"{'Board':<10} {'Babel best':<20} {'ELO':>6} {'Blank best':<20} {'ELO':>6}")
print('-'*70)
for board, data in tournament_results.items():
    elo = data['elo']
    bb = max([(n,e) for n,e in elo.items() if 'Babel' in n], key=lambda x:x[1], default=('—',0))
    bl = max([(n,e) for n,e in elo.items() if 'Blank' in n], key=lambda x:x[1], default=('—',0))
    print(f'{board:<10} {bb[0]:<20} {bb[1]:>6.0f} {bl[0]:<20} {bl[1]:>6.0f}')

In [ ]:
# === EXPORT ===
def cvt(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

export = {}
for board, data in tournament_results.items():
    export[board] = {
        'agents': data['agents'],
        'elo': {k: round(v, 1) for k, v in data['elo'].items()},
        'ranked': [(n, round(e, 1)) for n, e in data['ranked']],
        'matches': data['results'],
    }

with open('tournament_19_21.json', 'w') as f:
    json.dump(export, f, default=cvt, indent=2)

!cp tournament_19_21.json *.pdf "{ROOT}/"
print('\nExported:')
for f in sorted(glob.glob('*.pdf') + ['tournament_19_21.json']):
    print(f'  {f}')